# Dual Swin V2 — Late Fusion v8-late (`.tif` / phase1 dataset)

Adapted from source notebook to use `data/phase1_dataset/` GeoTIFFs (official repo).
Inference also supports `data/phase2_dataset/`.

```
RGB (3ch) → [ECA] → Swin V2 (intra SE) → Hybrid Decoder → [CBAM] → Head → logits_rgb
AUX (4ch) → [ECA] → Swin V2 (intra SE) → Hybrid Decoder → [CBAM] → Head → logits_aux
                                                                                ↓
                                              α·logits_rgb + (1-α)·logits_aux  (learnable α)
```

| Component | Mid-fusion v6 (0.8666) | v8-late |
|---|---|---|
| **Fusion strategy** | Mid: concat+ECA after encoders | **Late: blended logits** |
| **Decoders** | 1 shared | **2 separate (RGB + AUX)** |
| **Data format** | `.tif` GeoTIFF (7-band) | `.tif` GeoTIFF (7-band) |
| **Normalization** | Per-image P1/P99 + z-score | Same |
| Input attention | ECA | ECA |
| Intra-encoder | SE | SE |
| Decoder output | CBAM | CBAM (one per decoder) |

See `experiments/dual_swin_late_fusion_v8late_kfold_infer.ipynb` for inference.


In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "timm", "albumentations", "rasterio", "opencv-python-headless",
                       "tqdm", "scikit-learn", "matplotlib"])

In [ ]:

# ============================================================
# Dual-Swin V2 — LATE FUSION (v8-late)
# + Hybrid SegFormer×UNet++ Decoder (one per modality)
# K-Fold Cross-Validation — PHASE1 .tif DATASET (per-image norm)
# ============================================================

import os, json, time, zipfile, random, math, copy
from pathlib import Path

import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import albumentations as A
import rasterio
import timm
from sklearn.model_selection import KFold

# ============================================================
# Device & Reproducibility
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

# ============================================================
# Channel mapping
# ============================================================
CHANNEL_MAP = {
    "CTX": [0], "SLOPE": [1], "DEM": [2],
    "B3": [3], "B4": [4], "B5": [5], "B6": [6],
}
RGB_INDICES = [4, 5, 6]
AUX_INDICES = [0, 1, 2, 3]
ALL_INDICES = list(range(7))
BAND_INDICES_ALL = list(range(1, 8))  # rasterio 1-based bands 1-7

EXP = {"id": "DualSwinV2_LateFusion_MultiAttn_v8late",
       "channels": "all 7 merged/standardized"}

# ============================================================
# Augmentations
# ============================================================
def build_transforms(img_size, strong=True):
    if strong:
        geo_aug = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                scale=(0.85, 1.15),
                rotate=(-25, 25),
                p=0.5,
                border_mode=cv2.BORDER_REFLECT_101,
            ),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),
            A.CoarseDropout(
                num_holes_range=(2, 6),
                hole_height_range=(int(img_size * 0.04), int(img_size * 0.08)),
                hole_width_range=(int(img_size * 0.04), int(img_size * 0.08)),
                fill=0, p=0.2,
            ),
        ])
    else:
        geo_aug = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                scale=(0.90, 1.10),
                rotate=(-20, 20),
                p=0.5,
                border_mode=cv2.BORDER_REFLECT_101,
            ),
        ])
    rgb_photo = A.Compose([
        A.GaussianBlur(p=0.15),
        A.RandomBrightnessContrast(p=0.3),
    ])
    val_aug = A.Compose([A.Resize(img_size, img_size)])
    return geo_aug, rgb_photo, val_aug

# ============================================================
# Normalization stats (kept for reference; per-image norm used in training)
# ============================================================
def load_channel_stats(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)
    mean_all = np.array(stats["mean"], dtype=np.float32)
    std_all  = np.array(stats["std"],  dtype=np.float32)
    print(f"Loaded channel stats from {stats_path}")
    print(f"  mean = {mean_all}")
    print(f"  std  = {std_all}")
    return mean_all, std_all

# ============================================================
# Per-image percentile normalization (domain-invariant, v6-DA)
# ============================================================
def normalize_bands_per_image(arr_chw, p_low=1.0, p_high=99.0):
    """Per-image, per-band percentile normalization.
    Each band is clipped to its OWN [P_low, P_high] percentiles
    and rescaled to [0, 1].  Eliminates sensitivity to absolute
    value shifts between domains (e.g. DEM at different elevations).
    """
    C = arr_chw.shape[0]
    out = np.empty_like(arr_chw, dtype=np.float32)
    for c in range(C):
        flat = arr_chw[c].ravel()
        lo = np.percentile(flat, p_low)
        hi = np.percentile(flat, p_high)
        hi = max(hi, lo + 1e-6)
        out[c] = np.clip((arr_chw[c].astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return out


def compute_mean_std_per_image_norm(img_paths, band_indices,
                                    p_low=1.0, p_high=99.0,
                                    max_files=None, max_pixels=2_000_000):
    """Compute channel mean/std AFTER per-image percentile normalisation."""
    paths = list(img_paths)
    if max_files is not None:
        paths = paths[:max_files]
    C = len(band_indices)
    sums = np.zeros(C, dtype=np.float64)
    sqs  = np.zeros(C, dtype=np.float64)
    n    = 0
    rng  = np.random.default_rng(123)
    for p in tqdm(paths, desc="Compute mean/std (per-image norm)"):
        with rasterio.open(str(p)) as src:
            arr = src.read(band_indices).astype(np.float32)
        arr = normalize_bands_per_image(arr, p_low, p_high)
        flat = arr.reshape(C, -1)
        if flat.shape[1] > 20000:
            idx = rng.choice(flat.shape[1], size=20000, replace=False)
            flat = flat[:, idx]
        sums += flat.sum(axis=1)
        sqs  += (flat * flat).sum(axis=1)
        n    += flat.shape[1]
        if n >= max_pixels:
            break
    means = (sums / max(n, 1)).astype(np.float32)
    vars_ = (sqs  / max(n, 1) - means.astype(np.float64) ** 2).clip(min=1e-8)
    stds  = np.sqrt(vars_).astype(np.float32)
    return means, stds


# ============================================================
# Dataset
# ============================================================
class MarsDualModalSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, rgb_indices, aux_indices, mean_all, std_all,
                 geo_aug=None, rgb_photo_aug=None, val_aug=None, is_train=True):
        self.img_paths = list(img_paths)
        self.mask_paths = list(mask_paths) if mask_paths is not None else None
        self.rgb_indices = rgb_indices
        self.aux_indices = aux_indices
        self.mean_all = mean_all
        self.std_all = std_all
        self.geo_aug = geo_aug
        self.rgb_photo_aug = rgb_photo_aug
        self.val_aug = val_aug
        self.is_train = is_train

    def __len__(self):
        return len(self.img_paths)

    def _standardize(self, x_chw, indices):
        mean = self.mean_all[indices][:, None, None]
        std  = self.std_all[indices][:, None, None]
        return (x_chw - mean) / std

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        with rasterio.open(str(img_path)) as src:
            arr = src.read().astype(np.float32)          # (7, H, W)
        arr = normalize_bands_per_image(arr)              # per-image P1/P99 → [0,1]
        arr_hwc = np.transpose(arr, (1, 2, 0))

        mask = None
        if self.mask_paths is not None:
            with rasterio.open(str(self.mask_paths[idx])) as src:
                mask = src.read(1).astype(np.float32)

        if self.is_train:
            if self.geo_aug is not None:
                aug = self.geo_aug(image=arr_hwc, mask=mask)
                arr_hwc = aug["image"]; mask = aug["mask"]
            if self.rgb_photo_aug is not None:
                rgb_ch = arr_hwc[..., self.rgb_indices]
                aux_ch = arr_hwc[..., self.aux_indices]
                rgb_ch = self.rgb_photo_aug(image=rgb_ch)["image"]
                out = np.empty_like(arr_hwc)
                for i, ci in enumerate(self.rgb_indices):
                    out[..., ci] = rgb_ch[..., i]
                for i, ci in enumerate(self.aux_indices):
                    out[..., ci] = aux_ch[..., i]
                arr_hwc = out
        else:
            if self.val_aug is not None:
                if mask is not None:
                    aug = self.val_aug(image=arr_hwc, mask=mask)
                    arr_hwc = aug["image"]; mask = aug["mask"]
                else:
                    arr_hwc = self.val_aug(image=arr_hwc)["image"]

        arr_chw = np.transpose(arr_hwc, (2, 0, 1))
        rgb = self._standardize(arr_chw[self.rgb_indices], self.rgb_indices)
        aux = self._standardize(arr_chw[self.aux_indices], self.aux_indices)
        rgb_t = torch.from_numpy(rgb).float()
        aux_t = torch.from_numpy(aux).float()
        if mask is not None:
            mask_t = torch.from_numpy(mask).float().unsqueeze(0)
            return rgb_t, aux_t, mask_t
        else:
            return rgb_t, aux_t, str(img_path)

# ============================================================
# Pos_weight
# ============================================================
def compute_pos_weight(mask_paths):
    fg = 0; tot = 0
    for p in tqdm(mask_paths, desc="Compute pos_weight"):
        with rasterio.open(str(p)) as src:
            m = src.read(1)
        m01 = (m > 0).astype(np.uint8)
        fg += int(m01.sum()); tot += int(m01.size)
    frac = fg / tot
    pos_weight = (1.0 - frac) / (frac + 1e-9)
    return float(frac), float(pos_weight)

# ============================================================
# Metrics
# ============================================================
@torch.no_grad()
def compute_leaderboard_metrics_from_loader(model, loader, thresh=0.5):
    model.eval()
    TP = FP = FN = TN = 0.0
    for rgb, aux, mask in tqdm(loader, desc="ValMetric", leave=False):
        rgb  = rgb.to(DEVICE, non_blocking=True)
        aux  = aux.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        out = model(rgb, aux)
        logits = out["logits"] if isinstance(out, dict) else out
        pred = (torch.sigmoid(logits) > thresh).float()
        TP += (pred * mask).sum().item()
        FP += (pred * (1 - mask)).sum().item()
        FN += ((1 - pred) * mask).sum().item()
        TN += ((1 - pred) * (1 - mask)).sum().item()
    eps = 1e-7
    iou_fg = TP / (TP + FP + FN + eps)
    iou_bg = TN / (TN + FP + FN + eps)
    miou   = 0.5 * (iou_fg + iou_bg)
    prec_fg = TP / (TP + FP + eps)
    rec_fg  = TP / (TP + FN + eps)
    f1_fg   = 2 * prec_fg * rec_fg / (prec_fg + rec_fg + eps)
    return {
        "IoU_fg": float(iou_fg), "IoU_bg": float(iou_bg), "mIoU": float(miou),
        "Precision_fg": float(prec_fg), "Recall_fg": float(rec_fg), "F1_fg": float(f1_fg),
        "TP": float(TP), "FP": float(FP), "FN": float(FN), "TN": float(TN),
    }

# ============================================================
# Loss: BCE + Dice + Boundary + Lovász (v6 recipe — unchanged)
# ============================================================
def dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2, 3))
    den = (probs + targets).sum(dim=(2, 3)) + eps
    return 1 - (num / den).mean()

def boundary_loss(logits, targets, kernel_size=3):
    pad = kernel_size // 2
    kernel = torch.ones(1, 1, kernel_size, kernel_size, device=targets.device)
    dilated = F.conv2d(targets, kernel, padding=pad).clamp(0, 1)
    eroded  = 1.0 - F.conv2d(1.0 - targets, kernel, padding=pad).clamp(0, 1)
    boundary_map = (dilated - eroded).clamp(0, 1)
    weight = 1.0 + 2.0 * boundary_map
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    return (bce * weight).mean()

def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    if len(labels) == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm = perm.data
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), grad)
    return loss

def lovasz_hinge(logits, labels):
    losses = []
    for log, lab in zip(logits.squeeze(1), labels.squeeze(1)):
        losses.append(lovasz_hinge_flat(log.reshape(-1), lab.reshape(-1)))
    return torch.stack(losses).mean()

class HybridBCEDiceBoundaryLoss(nn.Module):
    def __init__(self, pos_weight: float,
                 alpha=0.3, beta=0.3, gamma=0.2, delta=0.2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], device=DEVICE))
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.delta = delta

    def forward(self, logits, targets):
        l_bce  = self.bce(logits, targets)
        l_dice = dice_loss(logits, targets)
        l_bnd  = boundary_loss(logits, targets)
        l_lov  = lovasz_hinge(logits, targets)
        return (self.alpha * l_bce + self.beta * l_dice +
                self.gamma * l_bnd + self.delta * l_lov)

# ============================================================
# Deep Supervision loss wrapper (for late fusion: applied to each branch)
# ============================================================
class DeepSupervisionLoss(nn.Module):
    def __init__(self, base_loss_fn, aux_weight=0.3):
        super().__init__()
        self.base_loss = base_loss_fn
        self.aux_weight = aux_weight

    def forward(self, model_output, targets):
        if isinstance(model_output, dict):
            logits = model_output["logits"]
            main_loss = self.base_loss(logits, targets)
            aux_logits_list = model_output.get("aux_logits", [])
            if len(aux_logits_list) > 0:
                aux_loss = 0.0
                for aux_logits in aux_logits_list:
                    if aux_logits.shape[-2:] != targets.shape[-2:]:
                        aux_logits = F.interpolate(
                            aux_logits, size=targets.shape[-2:],
                            mode="bilinear", align_corners=False)
                    aux_loss += self.base_loss(aux_logits, targets)
                aux_loss /= len(aux_logits_list)
                return main_loss + self.aux_weight * aux_loss
            return main_loss
        else:
            return self.base_loss(model_output, targets)

class LateFusionDeepSupervisionLoss(nn.Module):
    """
    Loss for late fusion: supervise BOTH branches + the fused output.
    
    total = loss(fused_logits) + branch_weight * (loss(rgb_logits) + loss(aux_logits))
            + aux_weight * (deep_supervision on each branch's aux heads)
    """
    def __init__(self, base_loss_fn, branch_weight=0.3, aux_weight=0.3):
        super().__init__()
        self.base_loss = base_loss_fn
        self.branch_weight = branch_weight
        self.aux_weight = aux_weight

    def forward(self, model_output, targets):
        if not isinstance(model_output, dict):
            return self.base_loss(model_output, targets)

        # Main fused loss
        fused_logits = model_output["logits"]
        main_loss = self.base_loss(fused_logits, targets)

        total = main_loss

        # Branch-level supervision
        for key in ["rgb_logits", "aux_logits_branch"]:
            if key in model_output:
                branch_logits = model_output[key]
                if branch_logits.shape[-2:] != targets.shape[-2:]:
                    branch_logits = F.interpolate(
                        branch_logits, size=targets.shape[-2:],
                        mode="bilinear", align_corners=False)
                total = total + self.branch_weight * self.base_loss(branch_logits, targets)

        # Deep supervision aux heads (from UNet++ intermediate nodes)
        for key in ["rgb_aux", "aux_aux"]:
            aux_list = model_output.get(key, [])
            if len(aux_list) > 0:
                aux_loss = 0.0
                for a in aux_list:
                    if a.shape[-2:] != targets.shape[-2:]:
                        a = F.interpolate(a, size=targets.shape[-2:],
                                          mode="bilinear", align_corners=False)
                    aux_loss += self.base_loss(a, targets)
                aux_loss /= len(aux_list)
                total = total + self.aux_weight * aux_loss

        return total

# ============================================================
# EMA
# ============================================================
class EMA:
    def __init__(self, model: nn.Module, decay=0.995, warmup_steps=0):
        self.decay = decay
        self.warmup_steps = warmup_steps
        self.step_count = 0
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.clone()

    def _get_decay(self):
        if self.warmup_steps > 0 and self.step_count < self.warmup_steps:
            return min(self.decay, 1.0 - 1.0 / (self.step_count + 1))
        return self.decay

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self._get_decay()
        self.step_count += 1
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            assert name in self.shadow
            new_avg = (1.0 - d) * p.data + d * self.shadow[name]
            self.shadow[name] = new_avg.clone()

    def apply_shadow(self, model: nn.Module):
        self.backup = {}
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            self.backup[name] = p.data.clone()
            p.data = self.shadow[name].clone()

    def restore(self, model: nn.Module):
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            p.data = self.backup[name].clone()
        self.backup = {}

# ============================================================
# Backbone helpers
# ============================================================
def adapt_patch_embed_in_chans(model, in_chans_new):
    pe = model.patch_embed
    old_conv = pe.proj
    old_w = old_conv.weight.data
    embed_dim, old_in, kh, kw = old_w.shape
    assert old_in == 3, f"Expected 3ch pretrained, got {old_in}"
    new_conv = nn.Conv2d(
        in_chans_new, embed_dim,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=(old_conv.bias is not None),
    )
    with torch.no_grad():
        new_w = torch.zeros(embed_dim, in_chans_new, kh, kw, device=old_w.device)
        new_w[:, :3, :, :] = old_w
        if in_chans_new > 3:
            rgb_mean = old_w.mean(dim=1, keepdim=True)
            new_w[:, 3:, :, :] = rgb_mean.repeat(1, in_chans_new - 3, 1, 1)
        new_conv.weight.copy_(new_w)
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias.data)
    pe.proj = new_conv
    return model

def make_swin_features(encoder_name, pretrained=True, img_size=256):
    enc = timm.create_model(
        encoder_name,
        pretrained=pretrained,
        features_only=True,
        out_indices=(0, 1, 2, 3),
        img_size=img_size,
    )
    if hasattr(enc, 'patch_embed'):
        enc.patch_embed.img_size = None
        if hasattr(enc.patch_embed, 'strict_img_size'):
            enc.patch_embed.strict_img_size = False
    return enc

def to_nchw(feats, in_chs):
    out = []
    for f, c in zip(feats, in_chs):
        if f.ndim == 4 and f.shape[-1] == c and f.shape[1] != c:
            f = f.permute(0, 3, 1, 2).contiguous()
        out.append(f)
    return out

# ============================================================
# Common building blocks
# ============================================================
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x)

# ============================================================
# Channel Attention Modules: SE, ECA, CBAM
# ============================================================
class SE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x)

class ECA(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False),
            nn.Sigmoid(),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(F.adaptive_avg_pool2d(x, 1))
        max_out = self.mlp(F.adaptive_max_pool2d(x, 1))
        ca = self.sigmoid(avg_out + max_out)
        x = x * ca
        avg_s = x.mean(dim=1, keepdim=True)
        max_s, _ = x.max(dim=1, keepdim=True)
        sa = self.spatial(torch.cat([avg_s, max_s], dim=1))
        return x * sa

# ============================================================
# Attention wrappers
# ============================================================
class InputChannelAttention(nn.Module):
    def __init__(self, in_channels, attn_type='eca', reduction=16):
        super().__init__()
        if attn_type.lower() == 'se':
            self.attn = SE(in_channels, reduction)
        elif attn_type.lower() == 'eca':
            self.attn = ECA(in_channels)
        elif attn_type.lower() == 'cbam':
            self.attn = CBAM(in_channels, reduction)
        else:
            self.attn = nn.Identity()
    def forward(self, x):
        return self.attn(x)

class DecoderOutputAttention(nn.Module):
    def __init__(self, channels, attn_type='cbam', reduction=16):
        super().__init__()
        if attn_type.lower() == 'se':
            self.attn = SE(channels, reduction)
        elif attn_type.lower() == 'eca':
            self.attn = ECA(channels)
        elif attn_type.lower() == 'cbam':
            self.attn = CBAM(channels, reduction)
        else:
            self.attn = nn.Identity()
    def forward(self, x):
        return self.attn(x)

class SwinWithIntraAttention(nn.Module):
    """Intra-encoder attention via hooks. Applies SE/ECA/CBAM
    after each Swin stage independently per stream."""
    def __init__(self, base_encoder, attn_type='se', reduction=16):
        super().__init__()
        self.base_encoder = base_encoder
        self.attn_type = attn_type.lower()
        self.feature_info = base_encoder.feature_info
        self.chs = base_encoder.feature_info.channels()
        if self.attn_type == 'none':
            self.stage_attns = nn.ModuleList([nn.Identity() for _ in self.chs])
        elif self.attn_type == 'se':
            self.stage_attns = nn.ModuleList([SE(c, reduction) for c in self.chs])
        elif self.attn_type == 'eca':
            self.stage_attns = nn.ModuleList([ECA(c) for c in self.chs])
        elif self.attn_type == 'cbam':
            self.stage_attns = nn.ModuleList([CBAM(c, reduction) for c in self.chs])
        else:
            raise ValueError(f"Unknown intra-encoder attention: {attn_type}")
        self._hook_handles = []
        self._register_hooks()

    def _get_stage_modules(self):
        stages = []
        if hasattr(self.base_encoder, 'stages'):
            for i in range(len(self.base_encoder.stages)):
                stages.append(self.base_encoder.stages[i])
        elif hasattr(self.base_encoder, 'layers'):
            for i in range(len(self.base_encoder.layers)):
                stages.append(self.base_encoder.layers[i])
        return stages

    def _register_hooks(self):
        if self.attn_type == 'none':
            return
        stages = self._get_stage_modules()
        for i, stage in enumerate(stages):
            handle = stage.register_forward_hook(self._make_hook(i))
            self._hook_handles.append(handle)

    def _make_hook(self, stage_idx):
        def hook(module, input, output):
            if output.ndim == 4:
                c = self.chs[stage_idx]
                if output.shape[-1] == c and output.shape[1] != c:
                    out_nchw = output.permute(0, 3, 1, 2).contiguous()
                    out_attn = self.stage_attns[stage_idx](out_nchw)
                    output = out_attn.permute(0, 2, 3, 1).contiguous()
                else:
                    output = self.stage_attns[stage_idx](output)
            return output
        return hook

    def forward(self, x):
        return self.base_encoder(x)

    def __getattr__(self, name):
        if name in ['base_encoder', 'attn_type', 'chs', 'stage_attns',
                     '_hook_handles', 'feature_info']:
            return super().__getattr__(name)
        return getattr(self.base_encoder, name)

def make_swin_with_intra_attention(encoder_name, pretrained=True, img_size=256,
                                    intra_attn='se', reduction=16):
    base_enc = make_swin_features(encoder_name, pretrained=pretrained, img_size=img_size)
    if intra_attn.lower() == 'none':
        return base_enc
    return SwinWithIntraAttention(base_enc, attn_type=intra_attn, reduction=reduction)

# ============================================================
# Hybrid SegFormer × UNet++ Decoder
# ============================================================
class HybridSegFormerUNetPPDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        C = fpn_channels
        self.mlp_projs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, C, 1, bias=False),
                nn.BatchNorm2d(C),
                nn.ReLU(inplace=True),
            ) for c in in_channels_list
        ])
        self.seg_fuse = nn.Sequential(
            nn.Conv2d(C * 4, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )
        def _node(n_cat):
            return nn.Sequential(ConvBNReLU(C * n_cat, C), ConvBNReLU(C, C))
        self.x01 = _node(2); self.x11 = _node(2); self.x21 = _node(2)
        self.x02 = _node(3); self.x12 = _node(3)
        self.x03 = _node(4)
        self.upp_final = nn.Sequential(ConvBNReLU(C, C), nn.Dropout2d(0.1))
        self.gate_fuse = nn.Sequential(
            nn.Conv2d(2 * C, C, 1, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True),
        )
        self.gate_se = SEBlock(C, r=16)
        self.aux_head_01 = nn.Conv2d(C, 1, 1)
        self.aux_head_02 = nn.Conv2d(C, 1, 1)

    @staticmethod
    def _up(x, target):
        return F.interpolate(x, size=target.shape[-2:],
                             mode="bilinear", align_corners=False)

    def forward(self, feats):
        x00 = self.mlp_projs[0](feats[0])
        x10 = self.mlp_projs[1](feats[1])
        x20 = self.mlp_projs[2](feats[2])
        x30 = self.mlp_projs[3](feats[3])
        target_size = x00.shape[-2:]
        seg_global = self.seg_fuse(torch.cat([
            x00,
            F.interpolate(x10, size=target_size, mode="bilinear", align_corners=False),
            F.interpolate(x20, size=target_size, mode="bilinear", align_corners=False),
            F.interpolate(x30, size=target_size, mode="bilinear", align_corners=False),
        ], dim=1))
        x21 = self.x21(torch.cat([x20, self._up(x30, x20)], dim=1))
        x11 = self.x11(torch.cat([x10, self._up(x20, x10)], dim=1))
        x01 = self.x01(torch.cat([x00, self._up(x10, x00)], dim=1))
        x12 = self.x12(torch.cat([x10, x11, self._up(x21, x10)], dim=1))
        x02 = self.x02(torch.cat([x00, x01, self._up(x11, x00)], dim=1))
        x03 = self.x03(torch.cat([x00, x01, x02, self._up(x12, x00)], dim=1))
        upp_out = self.upp_final(x03)
        fused = self.gate_fuse(torch.cat([upp_out, seg_global], dim=1))
        fused = self.gate_se(fused)
        aux_list = [self.aux_head_01(x01), self.aux_head_02(x02)]
        return fused, aux_list

# ============================================================
# ★ LATE FUSION MODEL ★
# ============================================================
class DualSwinLateFusionSeg(nn.Module):
    """
    Late fusion: each modality has its OWN encoder + decoder + head.
    Final prediction = learnable-weighted average of both branches' logits.

    RGB path: input_eca → swin+intra_se → decoder → cbam → head → rgb_logits
    AUX path: input_eca → swin+intra_se → decoder → cbam → head → aux_logits
    Output:   alpha * rgb_logits + (1-alpha) * aux_logits
    """
    def __init__(self, encoder_name, pretrained, img_size, fpn_channels,
                 input_attn="eca", intra_encoder_attn="se",
                 decoder_output_attn="cbam"):
        super().__init__()

        # --- RGB branch ---
        self.rgb_input_attn = InputChannelAttention(3, attn_type=input_attn)
        self.enc_rgb = make_swin_with_intra_attention(
            encoder_name, pretrained=pretrained, img_size=img_size,
            intra_attn=intra_encoder_attn)
        self.chs = self.enc_rgb.feature_info.channels()
        self.dec_rgb = HybridSegFormerUNetPPDecoder(self.chs, fpn_channels)
        self.dec_rgb_attn = DecoderOutputAttention(fpn_channels, attn_type=decoder_output_attn)
        self.head_rgb = nn.Conv2d(fpn_channels, 1, 1)

        # --- AUX branch ---
        self.aux_input_attn = InputChannelAttention(4, attn_type=input_attn)
        self.enc_aux = make_swin_with_intra_attention(
            encoder_name, pretrained=pretrained, img_size=img_size,
            intra_attn=intra_encoder_attn)
        # Adapt patch embed for 4 channels
        if intra_encoder_attn.lower() == 'none':
            adapt_patch_embed_in_chans(self.enc_aux, 4)
        else:
            adapt_patch_embed_in_chans(self.enc_aux.base_encoder, 4)
        self.dec_aux = HybridSegFormerUNetPPDecoder(self.chs, fpn_channels)
        self.dec_aux_attn = DecoderOutputAttention(fpn_channels, attn_type=decoder_output_attn)
        self.head_aux = nn.Conv2d(fpn_channels, 1, 1)

        # --- Learnable fusion weight (initialized to 0.5) ---
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))  # sigmoid(0) = 0.5

        self.img_size = img_size

    def _encode(self, encoder, x):
        feats = encoder(x)
        return to_nchw(feats, self.chs)

    def _decode_branch(self, decoder, dec_attn, head, feats):
        x, aux_list = decoder(feats)
        x = dec_attn(x)
        logits = head(x)
        logits = F.interpolate(logits, size=(self.img_size, self.img_size),
                               mode="bilinear", align_corners=False)
        aux_logits = [F.interpolate(a, size=(self.img_size, self.img_size),
                                    mode="bilinear", align_corners=False)
                      for a in aux_list]
        return logits, aux_logits

    def forward(self, rgb, aux4):
        # RGB branch
        rgb_in = self.rgb_input_attn(rgb)
        rgb_feats = self._encode(self.enc_rgb, rgb_in)
        rgb_logits, rgb_aux = self._decode_branch(
            self.dec_rgb, self.dec_rgb_attn, self.head_rgb, rgb_feats)

        # AUX branch
        aux_in = self.aux_input_attn(aux4)
        aux_feats = self._encode(self.enc_aux, aux_in)
        aux_logits, aux_aux = self._decode_branch(
            self.dec_aux, self.dec_aux_attn, self.head_aux, aux_feats)

        # Late fusion: learnable weighted average
        alpha = torch.sigmoid(self.alpha_logit)
        fused_logits = alpha * rgb_logits + (1 - alpha) * aux_logits

        if self.training:
            return {
                "logits": fused_logits,
                "rgb_logits": rgb_logits,
                "aux_logits_branch": aux_logits,
                "rgb_aux": rgb_aux,
                "aux_aux": aux_aux,
            }
        else:
            return fused_logits

# ============================================================
# Training / Validation
# ============================================================
@torch.no_grad()
def validate_loss(model, loader, loss_fn, use_amp=True):
    model.eval()
    total = 0.0; n = 0
    for rgb, aux, mask in tqdm(loader, desc="ValLoss", leave=False):
        rgb  = rgb.to(DEVICE, non_blocking=True)
        aux  = aux.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(use_amp and DEVICE == "cuda")):
            out = model(rgb, aux)
            logits = out["logits"] if isinstance(out, dict) else out
            loss = loss_fn(logits, mask)
        total += loss.item() * rgb.size(0); n += rgb.size(0)
    return total / max(n, 1)

# ============================================================
# TTA
# ============================================================
@torch.no_grad()
def tta_predict(model, rgb, aux):
    model.eval()
    probs_sum = None
    transforms = [
        lambda r, a: (r, a),
        lambda r, a: (r.flip(-1), a.flip(-1)),
        lambda r, a: (r.flip(-2), a.flip(-2)),
        lambda r, a: (r.flip(-1).flip(-2), a.flip(-1).flip(-2)),
    ]
    inv_transforms = [
        lambda p: p,
        lambda p: p.flip(-1),
        lambda p: p.flip(-2),
        lambda p: p.flip(-1).flip(-2),
    ]
    for tfm, inv in zip(transforms, inv_transforms):
        r, a = tfm(rgb, aux)
        out = model(r, a)
        logits = out["logits"] if isinstance(out, dict) else out
        probs = inv(torch.sigmoid(logits))
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(transforms)

# ============================================================
# Submission writing
# ============================================================
def _write_mask_tiff(mask01, out_path, height=128, width=128):
    if mask01.shape[0] != height or mask01.shape[1] != width:
        mask01 = cv2.resize(
            mask01.astype(np.uint8), (width, height),
            interpolation=cv2.INTER_NEAREST,
        )
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=height, width=width,
        count=1, dtype=rasterio.uint8,
    ) as dst:
        dst.write(mask01.astype(np.uint8), 1)

@torch.no_grad()
def ensemble_predict_tiffs_tta(models, loader, out_dir, thresh=0.5,
                                orig_size=128, use_tta=True):
    out_dir.mkdir(parents=True, exist_ok=True)
    for m in models:
        m.eval()
    for rgb, aux, src_paths in tqdm(loader, desc="Ensemble+TTA Infer", leave=False):
        rgb = rgb.to(DEVICE, non_blocking=True)
        aux = aux.to(DEVICE, non_blocking=True)
        avg_probs = None
        for m in models:
            if use_tta:
                probs = tta_predict(m, rgb, aux)
            else:
                out = m(rgb, aux)
                logits = out["logits"] if isinstance(out, dict) else out
                probs = torch.sigmoid(logits)
            avg_probs = probs if avg_probs is None else avg_probs + probs
        avg_probs = (avg_probs / len(models)).cpu().numpy()
        for i in range(avg_probs.shape[0]):
            mask01 = (avg_probs[i, 0] > thresh).astype(np.uint8)
            npy_name = Path(src_paths[i]).stem
            out_path = out_dir / f"{npy_name}.tif"
            _write_mask_tiff(mask01, str(out_path), height=orig_size, width=orig_size)

def zip_submission(pred_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for tif_path in sorted(pred_dir.glob("*.tif")):
            zf.write(tif_path, arcname=tif_path.name)

# ============================================================
# K-Fold CV Runner — v8-late
# ============================================================
def run_kfold_experiment(cfg, all_img_paths, all_mask_paths, test_img_paths,
                         means, stds, pos_weight):
    out_dir  = Path(cfg["out_dir"])
    tag      = "swinv2_hybrid_lateFusion_v8late"
    ckpt_dir = out_dir / "checkpoints" / tag
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    n_folds = cfg["n_folds"]
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=cfg["seed"])
    geo_aug, rgb_photo_aug, val_aug = build_transforms(
        cfg["img_size"], strong=cfg.get("strong_aug", True))

    fold_results = []
    fold_models = []

    warmup_epochs = cfg.get("warmup_epochs", 3)
    grad_clip     = cfg.get("grad_clip", 1.0)

    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(all_img_paths)):
        fold_num = fold_idx + 1
        print(f"\n--- Fold {fold_num}/{n_folds} ---")
        print(f"    Train: {len(train_indices)} | Val: {len(val_indices)}")

        fold_train_imgs  = [all_img_paths[i] for i in train_indices]
        fold_train_masks = [all_mask_paths[i] for i in train_indices]
        fold_val_imgs    = [all_img_paths[i] for i in val_indices]
        fold_val_masks   = [all_mask_paths[i] for i in val_indices]

        train_ds = MarsDualModalSegDataset(
            fold_train_imgs, fold_train_masks,
            RGB_INDICES, AUX_INDICES, means, stds,
            geo_aug=geo_aug, rgb_photo_aug=rgb_photo_aug, val_aug=None, is_train=True)
        val_ds = MarsDualModalSegDataset(
            fold_val_imgs, fold_val_masks,
            RGB_INDICES, AUX_INDICES, means, stds,
            geo_aug=None, rgb_photo_aug=None, val_aug=val_aug, is_train=False)

        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                                  num_workers=cfg["num_workers"], pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                                  num_workers=cfg["num_workers"], pin_memory=True)

        steps_per_epoch = len(train_loader)
        warmup_steps = warmup_epochs * steps_per_epoch

        model = DualSwinLateFusionSeg(
            encoder_name=cfg["encoder"],
            pretrained=cfg["pretrained"],
            img_size=cfg["img_size"],
            fpn_channels=cfg["fpn_channels"],
            input_attn=cfg.get("input_attn", "eca"),
            intra_encoder_attn=cfg.get("intra_encoder_attn", "se"),
            decoder_output_attn=cfg.get("decoder_output_attn", "cbam"),
        ).to(DEVICE)

        if fold_num == 1:
            n_params = sum(p.numel() for p in model.parameters())
            print(f"    Model params: {n_params:,}")

        base_loss_fn = HybridBCEDiceBoundaryLoss(pos_weight=pos_weight)
        loss_fn = LateFusionDeepSupervisionLoss(
            base_loss_fn,
            branch_weight=cfg.get("branch_weight", 0.3),
            aux_weight=cfg.get("aux_weight", 0.3))

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                      weight_decay=cfg["weight_decay"])

        _warmup_iters = warmup_epochs * steps_per_epoch
        _total_iters  = cfg["epochs"] * steps_per_epoch
        def lr_lambda(step, _wi=_warmup_iters, _ti=_total_iters):
            if step < _wi:
                return max(step / max(_wi, 1), 0.01)
            progress = (step - _wi) / max(_ti - _wi, 1)
            return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

        scaler = torch.amp.GradScaler("cuda", enabled=(cfg["amp"] and DEVICE == "cuda"))
        ema    = EMA(model, decay=cfg["ema_decay"], warmup_steps=warmup_steps)

        best_miou  = -1.0
        best_epoch = -1
        best_ckpt  = str(ckpt_dir / f"fold{fold_num}_best.pt")
        epoch_logs = []

        for epoch in range(1, cfg["epochs"] + 1):
            model.train()
            total_train_loss = 0.0; n_train = 0
            for rgb, aux, mask in tqdm(train_loader, desc="Train", leave=False):
                rgb  = rgb.to(DEVICE, non_blocking=True)
                aux  = aux.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=(cfg["amp"] and DEVICE == "cuda")):
                    out = model(rgb, aux)
                    loss = loss_fn(out, mask)
                if scaler is not None and (cfg["amp"] and DEVICE == "cuda"):
                    scaler.scale(loss).backward()
                    if grad_clip > 0:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if grad_clip > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    optimizer.step()
                scheduler.step()
                ema.update(model)
                bs = rgb.size(0)
                total_train_loss += loss.item() * bs; n_train += bs
            train_loss = total_train_loss / max(n_train, 1)

            model.eval()
            raw_val_loss = validate_loss(model, val_loader, base_loss_fn, use_amp=cfg["amp"])
            raw_metrics  = compute_leaderboard_metrics_from_loader(
                model, val_loader, thresh=cfg["thresh"])

            ema.apply_shadow(model)
            ema_val_loss = validate_loss(model, val_loader, base_loss_fn, use_amp=cfg["amp"])
            ema_metrics  = compute_leaderboard_metrics_from_loader(
                model, val_loader, thresh=cfg["thresh"])
            ema.restore(model)

            if ema_metrics["mIoU"] >= raw_metrics["mIoU"]:
                val_loss = ema_val_loss; metrics = ema_metrics; use_ema = True
            else:
                val_loss = raw_val_loss; metrics = raw_metrics; use_ema = False

            lr_now = float(optimizer.param_groups[0]["lr"])
            alpha_val = float(torch.sigmoid(model.alpha_logit).item())

            log_entry = {
                "epoch": epoch, "train_loss": float(train_loss),
                "val_loss": float(val_loss), "lr": lr_now, "used_ema": use_ema,
                "raw_mIoU": float(raw_metrics["mIoU"]),
                "ema_mIoU": float(ema_metrics["mIoU"]),
                "alpha_rgb": alpha_val,
                **{k: float(metrics[k]) for k in
                   ["mIoU", "IoU_fg", "IoU_bg", "F1_fg", "Precision_fg", "Recall_fg"]},
            }
            epoch_logs.append(log_entry)

            ema_tag = "EMA" if use_ema else "RAW"
            print(f"    Ep {epoch:3d}/{cfg['epochs']} | "
                  f"trn={train_loss:.4f} val={val_loss:.4f} | "
                  f"mIoU={metrics['mIoU']:.4f} F1={metrics['F1_fg']:.4f} "
                  f"IoU_fg={metrics['IoU_fg']:.4f} | "
                  f"[{ema_tag}] α_rgb={alpha_val:.3f}")

            if metrics["mIoU"] > best_miou:
                best_miou  = metrics["mIoU"]
                best_epoch = epoch
                if use_ema:
                    ema.apply_shadow(model)
                torch.save({
                    "model": model.state_dict(),
                    "fold": fold_num,
                    "best_epoch": best_epoch,
                    "best_metrics": metrics,
                    "used_ema": use_ema,
                    "alpha_rgb": alpha_val,
                }, best_ckpt)
                if use_ema:
                    ema.restore(model)

        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        fold_models.append(model)

        fold_results.append({
            "fold": fold_num,
            "best_epoch": best_epoch,
            "best_ckpt": best_ckpt,
            "best_metrics": ckpt["best_metrics"],
            "alpha_rgb": ckpt.get("alpha_rgb", 0.5),
            "epoch_logs": epoch_logs,
            "num_train": len(train_indices),
            "num_val": len(val_indices),
        })
        print(f"    => Fold {fold_num} best mIoU={best_miou:.4f} @ epoch {best_epoch} "
              f"(α_rgb={ckpt.get('alpha_rgb', 0.5):.3f})")

    # Aggregate
    metric_keys = ["mIoU", "IoU_fg", "IoU_bg", "F1_fg", "Precision_fg", "Recall_fg"]
    agg = {}
    for k in metric_keys:
        vals = [fr["best_metrics"][k] for fr in fold_results]
        agg[k] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)),
                   "min": float(np.min(vals)), "max": float(np.max(vals)),
                   "per_fold": vals}

    test_ds = MarsDualModalSegDataset(
        test_img_paths, None,
        RGB_INDICES, AUX_INDICES, means, stds,
        geo_aug=None, rgb_photo_aug=None, val_aug=val_aug, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False,
                             num_workers=cfg["num_workers"], pin_memory=True)

    sub_dir = out_dir / "submissions" / tag
    ensemble_predict_tiffs_tta(fold_models, test_loader, sub_dir,
                               thresh=cfg["thresh"], orig_size=cfg["orig_size"],
                               use_tta=cfg.get("use_tta", True))
    zip_path = str(out_dir / "submissions" / f"{tag}_kfold_ensemble.zip")
    zip_submission(sub_dir, zip_path)

    return {
        "tag": tag,
        "n_folds": n_folds,
        "fold_results": fold_results,
        "aggregate_metrics": agg,
        "ensemble_submission_zip": zip_path,
        "num_params": sum(p.numel() for p in fold_models[0].parameters()),
        "per_fold_alpha": [fr["alpha_rgb"] for fr in fold_results],
    }

print("All definitions loaded (v8-late — Late Fusion, .tif data, per-image norm).")

In [ ]:

# ============================================================
# EXPERIMENT CONFIGURATION (v8-late)
# ============================================================

cfg = dict(
    # --- Data ---
    data_root   = "../../data/phase1_dataset",  # ★ official repo phase1 .tif data
    out_dir     = "../../kfold_results_v8late",

    # --- Architecture ---
    encoder     = "swinv2_small_window8_256",

    # --- Attention ---
    input_attn          = "eca",
    intra_encoder_attn  = "se",
    decoder_output_attn = "cbam",
    # Fusion: LATE (built into model — no mid-fusion module)

    # --- K-Fold ---
    n_folds     = 5,

    # --- Training ---
    img_size    = 128,  # native .tif resolution
    orig_size   = 128,
    epochs      = 60,
    batch_size  = 12,        # ★ reduced from 16 → 12 (2 decoders = more VRAM)
    num_workers = 2,
    lr          = 2e-4,
    weight_decay= 1e-4,
    seed        = 42,
    pretrained  = True,
    fpn_channels= 256,
    amp         = True,
    ema_decay   = 0.995,
    warmup_epochs = 3,
    thresh      = 0.5,

    # --- Late fusion specific ---
    branch_weight = 0.3,    # weight for per-branch loss supervision
    aux_weight    = 0.3,    # deep supervision weight
    strong_aug    = True,
    grad_clip     = 1.0,
    use_tta       = True,
)

print(f"★ v8-late: LATE FUSION")
print(f"  Each modality gets its OWN encoder + decoder + head")
print(f"  Fusion: learnable α·rgb_logits + (1-α)·aux_logits")
print(f"  Attention: input={cfg['input_attn']}, intra={cfg['intra_encoder_attn']}, "
      f"dec_out={cfg['decoder_output_attn']}")
print(f"  Loss: fused + {cfg['branch_weight']}×(rgb + aux) branches + "
      f"{cfg['aux_weight']}× deep_sup")
print(f"  batch_size={cfg['batch_size']} (reduced for 2× decoder VRAM)")
print(f"  K-Folds: {cfg['n_folds']} × {cfg['epochs']} epochs")
print(f"  Target: beat mid-fusion v6 mIoU=0.8666")

In [ ]:
# ============================================================
# RUN K-FOLD CROSS-VALIDATION (v8-late)
# ============================================================
set_seed(cfg["seed"])
print("DEVICE:", DEVICE)
print("CFG:", json.dumps({k: v for k, v in cfg.items()
                          if not isinstance(v, (list, np.ndarray))}, indent=2))

data_root = Path(cfg["data_root"])

train_img_dir  = data_root / "train" / "images"
train_mask_dir = data_root / "train" / "masks"
val_img_dir    = data_root / "val" / "images"
val_mask_dir   = data_root / "val" / "masks"
# Phase 2 test: use phase2_dataset if available, else fall back to phase1 test
_phase2_test = Path("../../data/phase2_dataset/test/images")
if _phase2_test.exists():
    test_img_dir = _phase2_test
    print(f"Using phase2 test data: {test_img_dir}")
else:
    test_img_dir = data_root / "test" / "images"
    print(f"Using phase1 test data: {test_img_dir}")

train_imgs = sorted(list(train_img_dir.glob("*.tif")))
val_imgs   = sorted(list(val_img_dir.glob("*.tif")))
all_img_paths = train_imgs + val_imgs

all_mask_paths = []
for img_p in all_img_paths:
    if str(img_p).startswith(str(train_img_dir)):
        mask_p = train_mask_dir / img_p.name
    else:
        mask_p = val_mask_dir / img_p.name
    assert mask_p.exists(), f"Mask not found: {mask_p}"
    all_mask_paths.append(mask_p)

test_img_paths = sorted(list(test_img_dir.glob("*.tif")))

print(f"Total labeled samples (train+val): {len(all_img_paths)}")
print(f"  - From train/: {len(train_imgs)}")
print(f"  - From val/:   {len(val_imgs)}")
print(f"Test images: {len(test_img_paths)}")

# Compute global mean/std from training+val images (per-image-normalised)
BAND_INDICES_ALL = list(range(1, 8))  # rasterio 1-based bands 1-7
means, stds = compute_mean_std_per_image_norm(
    all_img_paths, BAND_INDICES_ALL, p_low=1.0, p_high=99.0)

# Save norm stats for inference reuse
out_dir = Path(cfg["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
norm_stats = {
    "normalization": "per_image_percentile",
    "p_low": 1.0, "p_high": 99.0,
    "band_indices_all": BAND_INDICES_ALL,
    "rgb_indices": RGB_INDICES, "aux_indices": AUX_INDICES,
    "channel_means": means.tolist(), "channel_stds": stds.tolist(),
    "img_size": cfg["img_size"],
}
norm_stats_path = out_dir / "norm_stats_v8late.json"
norm_stats_path.write_text(json.dumps(norm_stats, indent=2))
print(f"Norm stats saved → {norm_stats_path}")
fg_frac, pos_weight = compute_pos_weight(all_mask_paths)
print(f"FG frac: {fg_frac:.6f} | pos_weight: {pos_weight:.2f}")

out_dir = Path(cfg["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

print(f"\n{'=' * 80}")
print(f"Running Late Fusion v8-late — {cfg['n_folds']}-Fold CV")
print(f"{'=' * 80}")

result = run_kfold_experiment(
    cfg, all_img_paths, all_mask_paths, test_img_paths,
    means, stds, pos_weight)

report_json = out_dir / "kfold_report_v8late.json"
payload = {
    "experiment_id": EXP["id"],
    "channels": EXP["channels"],
    "rgb_indices": RGB_INDICES,
    "aux_indices": AUX_INDICES,
    "pos_weight": pos_weight,
    "total_labeled_samples": len(all_img_paths),
    "n_folds": cfg["n_folds"],
    "cfg": {k: v for k, v in cfg.items() if not isinstance(v, np.ndarray)},
    "result": result,
}
report_json.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print(f"\nReport saved: {report_json}")

agg = result["aggregate_metrics"]
print("\n" + "=" * 80)
print("K-FOLD CV SUMMARY (v8-late — Late Fusion)")
print("=" * 80)
print(f"  mIoU:      {agg['mIoU']['mean']:.4f} ± {agg['mIoU']['std']:.4f}")
print(f"  IoU_fg:    {agg['IoU_fg']['mean']:.4f} ± {agg['IoU_fg']['std']:.4f}")
print(f"  IoU_bg:    {agg['IoU_bg']['mean']:.4f} ± {agg['IoU_bg']['std']:.4f}")
print(f"  F1_fg:     {agg['F1_fg']['mean']:.4f} ± {agg['F1_fg']['std']:.4f}")
print(f"  Precision: {agg['Precision_fg']['mean']:.4f} ± {agg['Precision_fg']['std']:.4f}")
print(f"  Recall:    {agg['Recall_fg']['mean']:.4f} ± {agg['Recall_fg']['std']:.4f}")
print(f"  Per-fold mIoU:  {agg['mIoU']['per_fold']}")
print(f"  Per-fold alpha: {result['per_fold_alpha']}")
print(f"  Submission: {result['ensemble_submission_zip']}")
print(f"  Params: {result['num_params']:,}")

print("\nBaselines: v4-mid=0.8685 | v6-mid=0.8666 | v7-mid=0.8649")
print("DONE.")

In [ ]:
# ============================================================
# Visualize (v8-late)
# ============================================================
import matplotlib.pyplot as plt

agg = result["aggregate_metrics"]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Per-fold mIoU
fold_mious = agg["mIoU"]["per_fold"]
folds_x = list(range(1, len(fold_mious) + 1))
ax = axes[0]
ax.bar(folds_x, fold_mious, color="steelblue", edgecolor="black", alpha=0.8)
ax.axhline(y=agg["mIoU"]["mean"], color="red", linestyle="--", linewidth=2,
           label=f"v8-late = {agg['mIoU']['mean']:.4f}")
ax.fill_between([0.5, len(folds_x) + 0.5],
                agg["mIoU"]["mean"] - agg["mIoU"]["std"],
                agg["mIoU"]["mean"] + agg["mIoU"]["std"],
                alpha=0.15, color="red")
ax.axhline(y=0.8685, color="green", linestyle=":", linewidth=1.5, label="mid-fusion v4 = 0.8685")
ax.axhline(y=0.8666, color="orange", linestyle=":", linewidth=1.5, label="v6(mid) = 0.8666")
ax.set_xlabel("Fold"); ax.set_ylabel("mIoU")
ax.set_title("Per-Fold mIoU (v8-late)")
ax.set_xticks(folds_x); ax.legend(fontsize=8)
ax.set_ylim(min(fold_mious) * 0.98, max(max(fold_mious), 0.87) * 1.02)

# 2. Training curves
ax = axes[1]
for fr in result["fold_results"]:
    epochs = [e["epoch"] for e in fr["epoch_logs"]]
    mious  = [e["mIoU"]  for e in fr["epoch_logs"]]
    ax.plot(epochs, mious, label=f"Fold {fr['fold']}", alpha=0.7)
ax.set_xlabel("Epoch"); ax.set_ylabel("mIoU")
ax.set_title("Validation mIoU per Epoch")
ax.legend(); ax.grid(True, alpha=0.3)

# 3. Alpha (RGB weight) evolution
ax = axes[2]
for fr in result["fold_results"]:
    epochs = [e["epoch"] for e in fr["epoch_logs"]]
    alphas = [e["alpha_rgb"] for e in fr["epoch_logs"]]
    ax.plot(epochs, alphas, label=f"Fold {fr['fold']}", alpha=0.7)
ax.axhline(y=0.5, color="black", linewidth=0.5, linestyle="--", label="Equal weight")
ax.set_xlabel("Epoch"); ax.set_ylabel("α (RGB weight)")
ax.set_title("Learned Fusion Weight α (RGB vs AUX)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(Path(cfg["out_dir"]) / "kfold_plot_v8late.png", dpi=150)
plt.show()

# Summary
print(f"\nα interpretation: α > 0.5 means RGB dominates, α < 0.5 means AUX dominates")
print(f"Per-fold learned α: {result['per_fold_alpha']}")
print(f"\nBaselines: v4-mid=0.8685 | v6-mid=0.8666")